# 02 - Feature Engineering

This notebook walks through what `src.data.features.engineer_features` does,
and visualizes how each engineered feature interacts with the Arrest target.

Engineered features produced:

- Temporal: `Hour`, `DayOfWeek`, `Month`, `Year`, `IsWeekend`, `IsNight`, `Season`
- Categorical encodings: `PrimaryType_enc`, `LocationDesc_enc`, `Season_enc`, `Domestic_enc`
- Spatial/admin (passed through): `District`, `Community Area`, `Latitude`, `Longitude`

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.features import engineer_features, get_feature_matrix

sns.set_theme(style="whitegrid")

df = pd.read_csv(repo_root / "data" / "processed" / "chicago_cleaned.csv")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print("Raw cleaned columns:", list(df.columns))
print("Raw shape:", df.shape)

df_feat = engineer_features(df, persist_encoders=False)
print("\nEngineered columns (new):", sorted(set(df_feat.columns) - set(df.columns)))
print("Engineered shape:", df_feat.shape)

## 1. Distribution of each engineered temporal feature

In [ ]:
temporal_cols = ["Hour", "DayOfWeek", "Month", "Year", "IsWeekend", "IsNight"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, temporal_cols):
    counts = df_feat[col].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color="#1f77b4")
    ax.set_title(f"{col} distribution")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 2. Arrest rate vs each engineered feature

If a feature shifts the arrest rate substantially across its values, it
carries predictive signal. A flat bar means the feature is uninformative on
its own (though it can still help via interactions).

In [ ]:
overall_rate = df_feat["Arrest"].mean()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, col in zip(axes.flat, ["Hour", "Season", "IsNight", "IsWeekend"]):
    group = df_feat.groupby(col)["Arrest"].mean().sort_index()
    ax.bar(group.index.astype(str), group.values, color="#2ca02c")
    ax.axhline(overall_rate, color="red", linestyle="--", label=f"Overall ({overall_rate:.2%})")
    ax.set_title(f"Arrest rate by {col}")
    ax.set_ylabel("Arrest rate")
    ax.legend()
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 3. Category-to-integer encoding mappings

`LabelEncoder` is applied to `Primary Type`, `Location Description`, and `Season`.
Show the mapping for each, and the cardinality the model has to deal with.

In [ ]:
def encoding_table(raw_col, enc_col, top_n=10):
    pairs = df_feat[[raw_col, enc_col]].drop_duplicates().sort_values(enc_col)
    return pairs.head(top_n)

print("Primary Type encoding (first 10):")
print(encoding_table("Primary Type", "PrimaryType_enc"))
print(f"\nCardinality: {df_feat['PrimaryType_enc'].nunique()} unique primary types")

print("\nLocation Description encoding (first 10):")
print(encoding_table("Location Description", "LocationDesc_enc"))
print(f"\nCardinality: {df_feat['LocationDesc_enc'].nunique()} unique locations")

print("\nSeason encoding:")
print(encoding_table("Season", "Season_enc", top_n=4))

## 4. Final feature matrix

`get_feature_matrix` produces the X/y used by `train_evaluate_all`.

In [ ]:
X, y, feature_cols = get_feature_matrix(df_feat)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Feature columns:", feature_cols)

print("\nDescribe (rounded):")
X.describe().round(3).T[["mean", "std", "min", "max"]]

## Feature Engineering Takeaways

- Temporal features expose strong hour-of-day and night/weekend patterns the
  raw timestamp would hide from scikit-learn models.
- `PrimaryType_enc` and `LocationDesc_enc` are high-cardinality integer codes;
  tree-based models handle them well, but Logistic Regression treats them as
  ordinal, which is a known limitation.
- The resulting feature matrix is dense numeric with no missing values, ready
  for StandardScaler and SMOTE in the classification pipeline.